# ABSA Indodax — FASE 2: Pelabelan dan Persiapan Dataset

**Versi V5.4.1 — Perbaikan Scope Fleiss’ Kappa dan Checkpoint Robust**

Struktur utama tetap: validasi annotator → Fleiss’ Kappa → Majority Voting → near-duplicate grouping → group split → ROS training-only → checkpoint.

Kappa aspek dihitung pada seluruh baris dengan tiga label aspek lengkap. Kappa sentimen dihitung hanya pada baris yang Majority Voting aspeknya termasuk empat aspek substantif. `tidak_relevan` tetap berada pada model aspek.


In [ ]:
# =========================================================
# 0. INSTALL LIBRARY — KOMPATIBEL DENGAN RUNTIME COLAB
# =========================================================
# PyTorch mengikuti runtime agar kompatibel dengan CUDA. scikit-learn tidak
# diturunkan paksa agar tidak bentrok dengan paket bawaan Colab.

%pip -q install --upgrade-strategy only-if-needed \
    "transformers>=4.44,<5" \
    "accelerate>=0.34,<2" \
    "wordcloud>=1.9.3,<2" \
    "openpyxl>=3.1.5,<4" \
    "xlsxwriter>=3.2,<4"


In [ ]:
# =========================================================
# 1. IMPORT LIBRARY, SEED, DAN KONFIGURASI UTAMA — V5.4
# =========================================================

import os
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import re
import json
import math
import random
import shutil
import warnings
import zlib
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import sklearn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


def set_global_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as exc:
        print("Peringatan determinisme:", exc)


set_global_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("SEED:", SEED)

# -----------------------------
# Workspace pipeline terpisah
# -----------------------------
USE_GOOGLE_DRIVE = True
PROJECT_NAME = "ABSA_Indodax_Pipeline_V5_4_4Fase"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
    except Exception as exc:
        print("Google Drive tidak dapat dimount. Workspace lokal digunakan:", exc)
        PROJECT_DIR = Path.cwd() / PROJECT_NAME
else:
    PROJECT_DIR = Path.cwd() / PROJECT_NAME

STAGE1_DIR = PROJECT_DIR / "01_data_preparation"
STAGE2_DIR = PROJECT_DIR / "02_annotation_processing"
STAGE3_DIR = PROJECT_DIR / "03_model_training_evaluation"
for path in [PROJECT_DIR, STAGE1_DIR, STAGE2_DIR, STAGE3_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = "indodax_reviews.xlsx"
OUT_DIR = STAGE2_DIR

# Tahap 1 dipertahankan agar preprocessing tidak perlu diulang.
# Tahap 2–3 dibersihkan secara default agar hasil lama tidak tercampur.
RESET_STAGE2 = False
RESET_OUTPUT_DIR = RESET_STAGE2
if RESET_OUTPUT_DIR and OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = OUT_DIR / "figures"
MODEL_DIR = OUT_DIR / "models"
for path in [OUT_DIR, FIG_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUT_DIR tahap aktif:", OUT_DIR)

# -----------------------------
# Label penelitian
# -----------------------------
# Empat aspek berikut adalah aspek substantif yang dianalisis.
RESEARCH_ASPECT_LABELS = [
    "antarmuka_pengguna",
    "keamanan",
    "transaksi",
    "layanan_pelanggan",
]

# tidak_relevan dipakai sebagai kelas operasional model aspek untuk mencegah
# seluruh ulasan umum dipaksa masuk ke salah satu dari empat aspek penelitian.
IRRELEVANT_ASPECT_LABEL = "tidak_relevan"
ASPECT_MODEL_LABELS = RESEARCH_ASPECT_LABELS + [IRRELEVANT_ASPECT_LABEL]

# Alias kompatibilitas untuk fungsi lama: sekarang berisi lima kelas model aspek.
ASPECT_LABELS = ASPECT_MODEL_LABELS
ANNOTATION_ASPECT_LABELS = ASPECT_MODEL_LABELS
ASPECT_LABELING_POLICY = "single_dominant_aspect"

SENTIMENT_LABELS = ["negatif", "netral", "positif"]

ASPECT_DISPLAY_LABELS = {
    "antarmuka_pengguna": "Antarmuka pengguna / UI-UX",
    "keamanan": "Keamanan akun dan transaksi",
    "transaksi": "Transaksi, termasuk biaya, deposit, withdraw, pajak, order, dan jual-beli",
    "layanan_pelanggan": "Layanan pelanggan / customer service",
    "tidak_relevan": "Tidak relevan / tidak membahas empat aspek penelitian",
}

ASPECT_DEFINITIONS = {
    "antarmuka_pengguna": "Tampilan, kemudahan penggunaan, navigasi, performa aplikasi, bug, crash, loading, update, dan pengalaman penggunaan.",
    "keamanan": "Keamanan akun, login, OTP, KYC, verifikasi, autentikasi, privasi, pembobolan, dan perlindungan dana.",
    "transaksi": "Deposit, withdraw, transfer, fee/biaya, pajak, saldo, order beli/jual, pending, gagal transaksi, dan kecepatan transaksi.",
    "layanan_pelanggan": "Customer service, admin, bantuan, komplain, tiket, email, respons, dan penyelesaian masalah.",
    "tidak_relevan": "Ulasan umum atau ulasan yang tidak membahas satu pun dari empat aspek penelitian. Kelas ini menjadi relevance gate pada model aspek dan tidak ikut analisis sentimen per aspek.",
}

SENTIMENT_DEFINITIONS = {
    "negatif": "Keluhan, kekecewaan, kritik, masalah, pengalaman buruk, atau ketidakpuasan.",
    "netral": "Informasi deskriptif, pertanyaan, pernyataan tanpa emosi kuat, atau polaritas tidak jelas.",
    "positif": "Pujian, kepuasan, rekomendasi, apresiasi, atau pengalaman baik.",
}

ASPECT_ALIASES = {
    "antarmuka_pengguna": "antarmuka_pengguna",
    "antarmuka": "antarmuka_pengguna",
    "ui": "antarmuka_pengguna",
    "ux": "antarmuka_pengguna",
    "user_interface": "antarmuka_pengguna",
    "pengalaman_pengguna": "antarmuka_pengguna",
    "tampilan": "antarmuka_pengguna",
    "keamanan": "keamanan",
    "security": "keamanan",
    "aman": "keamanan",
    "verifikasi": "keamanan",
    "kyc": "keamanan",
    "otp": "keamanan",
    "login": "keamanan",
    "transaksi": "transaksi",
    "biaya_transaksi": "transaksi",
    "biaya": "transaksi",
    "fee": "transaksi",
    "pajak": "transaksi",
    "withdraw": "transaksi",
    "withdrawal": "transaksi",
    "wd": "transaksi",
    "deposit": "transaksi",
    "depo": "transaksi",
    "transfer": "transaksi",
    "saldo": "transaksi",
    "order": "transaksi",
    "jual_beli": "transaksi",
    "jual beli": "transaksi",
    "transaction": "transaksi",
    "layanan_pelanggan": "layanan_pelanggan",
    "layanan": "layanan_pelanggan",
    "customer_service": "layanan_pelanggan",
    "customer service": "layanan_pelanggan",
    "cs": "layanan_pelanggan",
    "support": "layanan_pelanggan",
    "bantuan": "layanan_pelanggan",
    "tidak_relevan": "tidak_relevan",
    "tidak relevan": "tidak_relevan",
    "umum": "tidak_relevan",
    "lainnya": "tidak_relevan",
    "di_luar_aspek": "tidak_relevan",
    "di luar aspek": "tidak_relevan",
}

SENTIMENT_ALIASES = {
    "negatif": "negatif",
    "negative": "negatif",
    "minus": "negatif",
    "buruk": "negatif",
    "jelek": "negatif",
    "netral": "netral",
    "neutral": "netral",
    "biasa": "netral",
    "informasi": "netral",
    "positif": "positif",
    "positive": "positif",
    "plus": "positif",
    "baik": "positif",
    "bagus": "positif",
}

# -----------------------------
# Sampling, split, near-duplicate, dan balancing
# -----------------------------
TOTAL_REVIEW_TARGET = 80000
LABELED_SAMPLE_SIZE = 8000

TRAIN_RATIO = 0.60
VAL_RATIO = 0.20
TEST_RATIO = 0.20
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

USE_RATING_FOR_SAMPLING_DIVERSITY = True
ANNOTATION_TEMPLATE_INCLUDE_RATING = False

GROUP_SPLIT_N_FOLDS = 5
NEAR_DUP_SIMILARITY_THRESHOLD = 0.92
NEAR_DUP_N_NEIGHBORS = 5
NEAR_DUP_MAX_FEATURES = 30000

# Dua dataset latih dibuat terpisah:
# 1) model aspek: oversampling berdasarkan kelas aspek;
# 2) model sentimen: hanya data relevan dan oversampling berdasarkan joint aspect-sentiment.
ASPECT_ROS_TARGET_MIN = 300
ASPECT_ROS_TARGET_MAX = 600
SENTIMENT_JOINT_ROS_TARGET_MIN = 100
SENTIMENT_JOINT_ROS_TARGET_MAX = 150

MIN_CLASS_COUNT_FOR_STRATIFY = 5
LOW_SUPPORT_THRESHOLD = 30

# -----------------------------
# Model dan early stopping
# -----------------------------
MODEL_NAME = "indobenchmark/indobert-base-p1"
FREEZE_BERT = False
MODEL_SELECTION_METRIC = "val_macro_f1"
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 0.002

TUNING_CONFIGS = [
    {
        "config_name": "layer2_h128_do035_lr2e-5_len160_ep5",
        "max_len": 160,
        "batch_size": 8,
        "learning_rate": 2e-5,
        "dropout": 0.35,
        "lstm_hidden": 128,
        "lstm_layers": 2,
        "epochs": 5,
    },
    {
        "config_name": "layer3_h128_do035_lr2e-5_len160_ep5",
        "max_len": 160,
        "batch_size": 8,
        "learning_rate": 2e-5,
        "dropout": 0.35,
        "lstm_hidden": 128,
        "lstm_layers": 3,
        "epochs": 5,
    },
    {
        "config_name": "layer4_h128_do035_lr2e-5_len160_ep5",
        "max_len": 160,
        "batch_size": 8,
        "learning_rate": 2e-5,
        "dropout": 0.35,
        "lstm_hidden": 128,
        "lstm_layers": 4,
        "epochs": 5,
    },
]

DEBUG_FAST_RUN = False
DEBUG_N_ROWS = 300


# Provenance wajib diisi jujur pada Notebook Bagian 2 sebelum hasil final.
# Pilihan:
# - manual_human
# - ai_assisted_human_verified
# - synthetic_ai_pipeline_test
ANNOTATION_PROVENANCE = "manual_human"
ALLOWED_ANNOTATION_PROVENANCE = {
    "manual_human",
    "ai_assisted_human_verified",
    "synthetic_ai_pipeline_test",
}
assert ANNOTATION_PROVENANCE in ALLOWED_ANNOTATION_PROVENANCE


library_versions = {
    "python": os.sys.version,
    "torch": torch.__version__,
    "transformers": __import__("transformers").__version__,
    "sklearn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

run_config = {
    "pipeline_version": "V5.4.1_FASE2_FIX_KAPPA_ROBUST",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "seed": SEED,
    "library_versions": library_versions,
    "raw_data_path": RAW_DATA_PATH,
    "total_review_target": TOTAL_REVIEW_TARGET,
    "labeled_sample_size": LABELED_SAMPLE_SIZE,
    "research_aspect_labels": RESEARCH_ASPECT_LABELS,
    "aspect_model_labels": ASPECT_MODEL_LABELS,
    "irrelevant_aspect_label": IRRELEVANT_ASPECT_LABEL,
    "sentiment_labels": SENTIMENT_LABELS,
    "aspect_labeling_policy": ASPECT_LABELING_POLICY,
    "split_strategy": "StratifiedGroupKFold_5fold_near_duplicate_group",
    "near_duplicate_similarity_threshold": NEAR_DUP_SIMILARITY_THRESHOLD,
    "train_ratio": TRAIN_RATIO,
    "validation_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "aspect_ros_target_range": [ASPECT_ROS_TARGET_MIN, ASPECT_ROS_TARGET_MAX],
    "sentiment_joint_ros_target_range": [
        SENTIMENT_JOINT_ROS_TARGET_MIN,
        SENTIMENT_JOINT_ROS_TARGET_MAX,
    ],
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "low_support_threshold": LOW_SUPPORT_THRESHOLD,
    "model_name": MODEL_NAME,
    "freeze_bert": FREEZE_BERT,
    "model_selection_metric": MODEL_SELECTION_METRIC,
    "tuning_configs": TUNING_CONFIGS,
}
if "ANNOTATION_PROVENANCE" in globals():
    run_config["annotation_provenance"] = ANNOTATION_PROVENANCE

with open(OUT_DIR / "00_run_config.json", "w", encoding="utf-8") as file:
    json.dump(run_config, file, indent=2, ensure_ascii=False)

with open(OUT_DIR / "00_library_versions.json", "w", encoding="utf-8") as file:
    json.dump(library_versions, file, indent=2, ensure_ascii=False)

print("Konfigurasi tersimpan:", OUT_DIR / "00_run_config.json")


# Catatan Revisi Metodologi V5.4.1

- struktur Majority Voting, near-duplicate grouping, group split, dan ROS tidak diubah;
- Kappa aspek memakai seluruh data aspek lengkap;
- Kappa sentimen memakai subset Majority Aspect relevan;
- provenance harus diisi sesuai proses nyata;
- ZIP hanya dipakai jika folder tidak lengkap dan lolos pemeriksaan.


## 2. Fungsi bantu umum

Bagian ini berisi fungsi untuk menyimpan tabel, membuat grafik, dan menghitung Fleiss' Kappa.

In [ ]:
# =========================================================
# 2. FUNGSI BANTU UMUM
# =========================================================

def save_df(df, filename, index=False):
    path = OUT_DIR / filename
    df.to_excel(path, index=index)
    print("Saved:", path)
    return path


def save_json(obj, filename):
    path = OUT_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    print("Saved:", path)
    return path


def normalize_label_text(value):
    """Normalisasi teks label agar variasi spasi, huruf besar, dan tanda hubung lebih aman."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in ["", "nan", "none", "null", "-", "_"]:
        return np.nan
    text = re.sub(r"[\s\-/]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text if text else np.nan


def canonicalize_aspect_label(value):
    """Memetakan variasi label aspek ke label resmi penelitian."""
    label = normalize_label_text(value)
    if pd.isna(label):
        return np.nan
    label = ASPECT_ALIASES.get(label, label)
    return label if label in ANNOTATION_ASPECT_LABELS else np.nan


def canonicalize_sentiment_label(value):
    """Memetakan variasi label sentimen ke label resmi penelitian."""
    label = normalize_label_text(value)
    if pd.isna(label):
        return np.nan
    label = SENTIMENT_ALIASES.get(label, label)
    return label if label in SENTIMENT_LABELS else np.nan


def display_label_guide():
    """Menampilkan panduan label yang konsisten dengan proposal dan notebook."""
    aspect_guide = pd.DataFrame([
        {"aspect_label": k, "display_name": ASPECT_DISPLAY_LABELS[k], "definition": ASPECT_DEFINITIONS[k]}
        for k in ANNOTATION_ASPECT_LABELS
    ])
    sentiment_guide = pd.DataFrame([
        {"sentiment_label": k, "definition": SENTIMENT_DEFINITIONS[k]}
        for k in SENTIMENT_LABELS
    ])
    display(aspect_guide)
    display(sentiment_guide)
    return aspect_guide, sentiment_guide


def plot_count_series(series, title, xlabel, ylabel, filename, rotation=30):
    plt.figure(figsize=(9, 5))
    series.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()
    path = FIG_DIR / filename
    plt.savefig(path, dpi=200)
    plt.show()
    print("Saved figure:", path)



def plot_heatmap_table(table, title, filename, xlabel="Sentimen", ylabel="Aspek"):
    """
    Membuat heatmap dari tabel/crosstab dan menyimpannya ke folder FIG_DIR.
    Fungsi ini dipakai untuk visualisasi distribusi joint label aspek x sentimen.
    """
    data = table.copy()

    # Pastikan data numerik agar dapat diplot dengan imshow.
    data = data.apply(pd.to_numeric, errors="coerce").fillna(0)

    plt.figure(figsize=(8, 5))
    im = plt.imshow(data.values, aspect="auto")
    plt.title(title)
    plt.colorbar(im, label="Jumlah Data")

    plt.xticks(
        ticks=np.arange(len(data.columns)),
        labels=[str(c) for c in data.columns],
        rotation=30,
        ha="right",
    )
    plt.yticks(
        ticks=np.arange(len(data.index)),
        labels=[str(i) for i in data.index],
    )

    max_value = data.values.max() if data.values.size > 0 else 0
    threshold = max_value / 2 if max_value > 0 else 0

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data.iloc[i, j]
            text_value = f"{value:.0f}"
            plt.text(
                j,
                i,
                text_value,
                ha="center",
                va="center",
                color="white" if value > threshold else "black",
            )

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    path = FIG_DIR / filename
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved figure:", path)
    return path


def plot_confusion_matrix(cm, labels, title, filename):
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)

    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black",
            )

    plt.ylabel("Label Aktual")
    plt.xlabel("Label Prediksi")
    plt.tight_layout()
    path = FIG_DIR / filename
    plt.savefig(path, dpi=200)
    plt.show()
    print("Saved figure:", path)


def plot_learning_curve(history_df, title_prefix, filename_prefix):
    if history_df.empty:
        return

    metrics = [
        ("train_loss", "val_loss", "Loss"),
        ("train_acc", "val_acc", "Accuracy"),
        ("train_macro_f1", "val_macro_f1", "Macro F1"),
    ]

    for train_col, val_col, y_label in metrics:
        if train_col not in history_df.columns or val_col not in history_df.columns:
            continue
        plt.figure(figsize=(8, 5))
        plt.plot(history_df["epoch"], history_df[train_col], marker="o", label=train_col)
        plt.plot(history_df["epoch"], history_df[val_col], marker="o", label=val_col)
        plt.title(f"{title_prefix} - {y_label}")
        plt.xlabel("Epoch")
        plt.ylabel(y_label)
        plt.legend()
        plt.tight_layout()
        path = FIG_DIR / f"{filename_prefix}_{y_label.lower().replace(' ', '_')}.png"
        plt.savefig(path, dpi=200)
        plt.show()
        print("Saved figure:", path)


def fleiss_kappa(label_matrix, categories):
    """Menghitung Fleiss' Kappa untuk beberapa annotator.

    Parameters
    ----------
    label_matrix : list[list[str]]
        Setiap baris berisi label dari seluruh annotator untuk satu objek.
        Fungsi ini mengharuskan jumlah annotator sama pada setiap baris dan
        seluruh label termasuk dalam ``categories``.
    categories : list[str]
        Seluruh kategori yang mungkin pada proses anotasi.
    """
    if not label_matrix:
        return np.nan

    rows = []
    expected_raters = None
    for row_idx, labels in enumerate(label_matrix):
        if expected_raters is None:
            expected_raters = len(labels)
        if len(labels) != expected_raters:
            raise ValueError(
                f"Jumlah annotator tidak konsisten pada baris {row_idx}: "
                f"expected={expected_raters}, got={len(labels)}"
            )
        unknown = [label for label in labels if label not in categories]
        if unknown:
            raise ValueError(
                f"Label di luar kategori pada baris {row_idx}: {unknown}. "
                f"Kategori resmi: {categories}"
            )
        rows.append([labels.count(cat) for cat in categories])

    M = np.asarray(rows, dtype=float)
    if M.size == 0:
        return np.nan

    row_totals = M.sum(axis=1)
    if not np.allclose(row_totals, row_totals[0]):
        raise ValueError("Jumlah penilaian per objek tidak konsisten.")

    N = M.shape[0]
    n = float(row_totals[0])
    if n <= 1:
        return np.nan

    agreement_per_item = ((M ** 2).sum(axis=1) - n) / (n * (n - 1))
    observed_agreement = agreement_per_item.mean()
    category_proportions = M.sum(axis=0) / (N * n)
    expected_agreement = (category_proportions ** 2).sum()

    if abs(1 - expected_agreement) < 1e-12:
        return np.nan
    return (observed_agreement - expected_agreement) / (1 - expected_agreement)


def majority_vote(labels):
    c = Counter([x for x in labels if pd.notna(x)])
    if not c:
        return np.nan
    top = c.most_common()
    # Jika voting seri, tidak dipakai agar label final lebih aman.
    if len(top) > 1 and top[0][1] == top[1][1]:
        return np.nan
    return top[0][0]


In [ ]:
# =========================================================
# 2A. SANITY CHECK FUNGSI WAJIB
# =========================================================

required_functions = [
    "save_df",
    "plot_count_series",
    "plot_heatmap_table",
    "plot_confusion_matrix",
    "plot_learning_curve",
    "fleiss_kappa",
    "majority_vote",
]

missing_required_functions = [fn for fn in required_functions if fn not in globals() or not callable(globals()[fn])]
if missing_required_functions:
    raise NameError(f"Fungsi wajib belum tersedia: {missing_required_functions}")

print("Sanity check fungsi wajib: OK")


In [ ]:
# =========================================================
# LOAD CHECKPOINT TAHAP 1 DAN PATH ANNOTATOR EKSPLISIT
# =========================================================
import zipfile

def inspect_zip(zip_path):
    if not zip_path.exists(): return False, "tidak ditemukan"
    if not zipfile.is_zipfile(zip_path): return False, "bukan ZIP valid"
    try:
        with zipfile.ZipFile(zip_path,"r") as a: bad=a.testzip()
        return (False,f"anggota rusak: {bad}") if bad else (True,"valid")
    except (zipfile.BadZipFile,OSError) as exc:
        return False,f"gagal dibaca: {exc}"

def restore_stage_zip_if_needed(stage_dir, zip_name):
    required=[stage_dir/"01_standardized_reviews.xlsx",stage_dir/"02_clean_reviews.xlsx",stage_dir/"05_annotation_master_sample.xlsx"]
    if all(p.is_file() for p in required):
        print("Fase 1 menggunakan folder lengkap:",stage_dir); return
    candidates=[PROJECT_DIR/zip_name,Path('/content')/zip_name,Path.cwd()/zip_name,Path('/mnt/data')/zip_name]
    audit=[]; chosen=None
    for p in candidates:
        if not p.exists(): continue
        ok,reason=inspect_zip(p); audit.append({'path':str(p),'valid':ok,'reason':reason})
        if ok and chosen is None: chosen=p
    if audit: display(pd.DataFrame(audit))
    if chosen is None:
        raise FileNotFoundError(f"Fase 1 belum lengkap dan tidak ada ZIP valid. Hilang: {[str(p) for p in required if not p.is_file()]}")
    stage_dir.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(chosen,'r') as a: a.extractall(stage_dir)
    missing=[str(p) for p in required if not p.is_file()]
    if missing: raise FileNotFoundError(f"Restore selesai tetapi file masih hilang: {missing}")
    print("Checkpoint Fase 1 diekstrak dari:",chosen)

restore_stage_zip_if_needed(STAGE1_DIR,"stage1_checkpoint.zip")
work_df=pd.read_excel(STAGE1_DIR/"01_standardized_reviews.xlsx")
clean_df=pd.read_excel(STAGE1_DIR/"02_clean_reviews.xlsx")
annotation_master_df=pd.read_excel(STAGE1_DIR/"05_annotation_master_sample.xlsx")
ANNOTATOR_RESULTS_DIR=PROJECT_DIR/"annotator_results"; ANNOTATOR_RESULTS_DIR.mkdir(parents=True,exist_ok=True)
ANNOTATOR_RESULT_PATHS=[ANNOTATOR_RESULTS_DIR/f"05_annotator{i}_result.xlsx" for i in range(1,4)]
missing=[str(p) for p in ANNOTATOR_RESULT_PATHS if not p.exists()]
if missing:
    raise FileNotFoundError("File annotator belum lengkap di " + str(ANNOTATOR_RESULTS_DIR) + ":\n- " + "\n- ".join(Path(p).name for p in missing))
FINAL_LABEL_FILE=OUT_DIR/"06_final_majority_labels.xlsx"
print("Checkpoint Fase 1 berhasil dimuat:",work_df.shape,clean_df.shape,annotation_master_df.shape)
print("Folder annotator:",ANNOTATOR_RESULTS_DIR)
print("ANNOTATION_PROVENANCE:",ANNOTATION_PROVENANCE)
if ANNOTATION_PROVENANCE=="synthetic_ai_pipeline_test":
    print("PERINGATAN: hasil hanya pengujian pipeline sampai setiap baris diverifikasi manusia.")


## 6. Fleiss’ Kappa, Majority Voting, dan Dataset Berlabel

1. Validasi annotator.
2. Kappa aspek pada tiga label aspek lengkap.
3. Majority Voting aspek.
4. Kappa sentimen hanya pada Majority Aspect relevan.
5. Majority Voting sentimen dan label final.
6. `tidak_relevan` tetap pada model aspek; model sentimen hanya data relevan.

Scope Kappa sentimen tidak mengubah label final, split, atau ROS.


In [ ]:
# =========================================================
# 7. VALIDASI ANNOTATOR, FLEISS' KAPPA, DAN MAJORITY VOTING
# =========================================================

def normalize_id_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))
    return str(value).strip()


master_required = ["id", "review_clean"]
missing_master = [col for col in master_required if col not in annotation_master_df.columns]
if missing_master:
    raise ValueError(f"Master sample kurang kolom: {missing_master}")

master_df = annotation_master_df[master_required].copy()
master_df["id_key"] = master_df["id"].apply(normalize_id_value)
if master_df["id_key"].duplicated().any():
    raise ValueError("Master sample memiliki ID duplikat.")

annotator_audit_rows = []
ann_dfs = []

for annotator_index, file_path in enumerate(ANNOTATOR_RESULT_PATHS, start=1):
    print(f"Membaca annotator {annotator_index}: {file_path}")
    raw_df = pd.read_excel(file_path)

    required_cols = [
        "id",
        "review_clean",
        "aspect_annotator",
        "sentiment_annotator",
    ]
    missing_cols = [col for col in required_cols if col not in raw_df.columns]
    if missing_cols:
        raise ValueError(
            f"Annotator {annotator_index} kurang kolom {missing_cols}. "
            f"Kolom tersedia: {list(raw_df.columns)}"
        )

    df_ann = raw_df[required_cols].copy()
    df_ann["id_key"] = df_ann["id"].apply(normalize_id_value)

    duplicate_id_count = int(df_ann["id_key"].duplicated().sum())
    if duplicate_id_count:
        raise ValueError(
            f"Annotator {annotator_index} memiliki {duplicate_id_count} ID duplikat."
        )

    master_ids = set(master_df["id_key"])
    annotator_ids = set(df_ann["id_key"])
    missing_ids = sorted(master_ids - annotator_ids)
    extra_ids = sorted(annotator_ids - master_ids)

    if missing_ids or extra_ids or len(df_ann) != len(master_df):
        raise ValueError(
            f"Annotator {annotator_index} tidak identik dengan master sample. "
            f"rows={len(df_ann)}, expected={len(master_df)}, "
            f"missing_ids={len(missing_ids)}, extra_ids={len(extra_ids)}"
        )

    ordered = master_df.merge(
        df_ann.drop(columns=["id"]),
        on="id_key",
        how="left",
        validate="one_to_one",
        suffixes=("_master", "_annotator"),
    )

    text_master = (
        ordered["review_clean_master"]
        .fillna("")
        .astype(str)
        .str.strip()
    )
    text_annotator = (
        ordered["review_clean_annotator"]
        .fillna("")
        .astype(str)
        .str.strip()
    )
    text_mismatch_count = int((text_master != text_annotator).sum())
    if text_mismatch_count:
        mismatch_preview = ordered.loc[
            text_master != text_annotator,
            ["id_key", "review_clean_master", "review_clean_annotator"],
        ].head(10)
        display(mismatch_preview)
        raise ValueError(
            f"Annotator {annotator_index}: {text_mismatch_count} review_clean "
            "berbeda dari master sample."
        )

    aspect_raw_col = f"aspect_ann{annotator_index}_raw"
    sentiment_raw_col = f"sentiment_ann{annotator_index}_raw"
    aspect_col = f"aspect_ann{annotator_index}"
    sentiment_col = f"sentiment_ann{annotator_index}"

    ordered[aspect_raw_col] = ordered["aspect_annotator"]
    ordered[sentiment_raw_col] = ordered["sentiment_annotator"]
    ordered[aspect_col] = ordered["aspect_annotator"].apply(
        canonicalize_aspect_label
    )
    ordered[sentiment_col] = ordered["sentiment_annotator"].apply(
        canonicalize_sentiment_label
    )

    invalid_aspect = int(ordered[aspect_col].isna().sum())
    invalid_sentiment = int(ordered[sentiment_col].isna().sum())

    annotator_audit_rows.append({
        "annotator": annotator_index,
        "file_path": str(file_path),
        "rows": len(ordered),
        "duplicate_id_count": duplicate_id_count,
        "missing_id_count": len(missing_ids),
        "extra_id_count": len(extra_ids),
        "review_text_mismatch_count": text_mismatch_count,
        "invalid_or_empty_aspect_count": invalid_aspect,
        "invalid_or_empty_sentiment_count": invalid_sentiment,
    })

    ann_dfs.append(
        ordered[
            [
                "id_key",
                aspect_raw_col,
                sentiment_raw_col,
                aspect_col,
                sentiment_col,
            ]
        ].copy()
    )

annotator_validation_df = pd.DataFrame(annotator_audit_rows)
display(annotator_validation_df)
save_df(annotator_validation_df, "06_annotator_file_validation_audit.xlsx")

if (
    annotator_validation_df[
        [
            "duplicate_id_count",
            "missing_id_count",
            "extra_id_count",
            "review_text_mismatch_count",
            "invalid_or_empty_aspect_count",
            "invalid_or_empty_sentiment_count",
        ]
    ].to_numpy().sum()
    > 0
):
    raise ValueError(
        "Validasi annotator gagal. Perbaiki file sebelum menghitung Kappa."
    )

annotation_df = master_df.copy()
for ann_df in ann_dfs:
    annotation_df = annotation_df.merge(
        ann_df,
        on="id_key",
        how="left",
        validate="one_to_one",
    )

annotation_cols = [
    "aspect_ann1",
    "aspect_ann2",
    "aspect_ann3",
    "sentiment_ann1",
    "sentiment_ann2",
    "sentiment_ann3",
]

# =====================================================
# 1) FLEISS' KAPPA ASPEK SEBELUM MAJORITY VOTING
# =====================================================
aspect_annotation_cols=["aspect_ann1","aspect_ann2","aspect_ann3"]
sentiment_annotation_cols=["sentiment_ann1","sentiment_ann2","sentiment_ann3"]
all_annotation_complete_mask=annotation_df[annotation_cols].notna().all(axis=1)
kappa_input_df=annotation_df.loc[all_annotation_complete_mask].copy()
kappa_excluded_df=annotation_df.loc[~all_annotation_complete_mask].copy()
if kappa_input_df.empty: raise ValueError("Tidak ada baris lengkap untuk Majority Voting dan Kappa.")
aspect_complete_mask=annotation_df[aspect_annotation_cols].notna().all(axis=1)
aspect_kappa_input_df=annotation_df.loc[aspect_complete_mask].copy()
aspect_kappa_excluded_df=annotation_df.loc[~aspect_complete_mask].copy()
kappa_aspect=fleiss_kappa(aspect_kappa_input_df[aspect_annotation_cols].values.tolist(),ANNOTATION_ASPECT_LABELS)

# =====================================================
# 2) MAJORITY VOTING ASPEK DAN SCOPE KAPPA SENTIMEN
# =====================================================
voting_df=kappa_input_df.copy()
voting_df["aspect"]=voting_df[aspect_annotation_cols].apply(lambda r: majority_vote(list(r)),axis=1)
sentiment_scope_mask=voting_df["aspect"].isin(RESEARCH_ASPECT_LABELS)
sentiment_kappa_input_df=voting_df.loc[sentiment_scope_mask].copy()
if sentiment_kappa_input_df.empty: raise ValueError("Tidak ada ulasan relevan untuk Kappa sentimen.")
kappa_sentiment=fleiss_kappa(sentiment_kappa_input_df[sentiment_annotation_cols].values.tolist(),SENTIMENT_LABELS)
sentiment_rows_excluded_outside_scope=int((~sentiment_scope_mask).sum())

kappa_df=pd.DataFrame([
 {'label_type':'aspect','scope':'seluruh_baris_dengan_tiga_label_aspek_lengkap','fleiss_kappa':kappa_aspect,'rows_used':len(aspect_kappa_input_df),'rows_excluded_invalid_or_missing':len(aspect_kappa_excluded_df),'rows_excluded_outside_scope':0,'categories':', '.join(ANNOTATION_ASPECT_LABELS),'annotation_provenance':ANNOTATION_PROVENANCE},
 {'label_type':'sentiment','scope':'hanya_majority_aspect_relevan','fleiss_kappa':kappa_sentiment,'rows_used':len(sentiment_kappa_input_df),'rows_excluded_invalid_or_missing':len(kappa_excluded_df),'rows_excluded_outside_scope':sentiment_rows_excluded_outside_scope,'categories':', '.join(SENTIMENT_LABELS),'annotation_provenance':ANNOTATION_PROVENANCE}
])
display(kappa_df); save_df(kappa_df,"06_fleiss_kappa.xlsx")
if not kappa_excluded_df.empty: save_df(kappa_excluded_df,"06_kappa_excluded_invalid_or_missing.xlsx")
if not aspect_kappa_excluded_df.empty: save_df(aspect_kappa_excluded_df,"06_kappa_aspect_excluded_invalid_or_missing.xlsx")
sentiment_outside_scope_df=voting_df.loc[~sentiment_scope_mask].copy()
if not sentiment_outside_scope_df.empty: save_df(sentiment_outside_scope_df,"06_kappa_sentiment_excluded_outside_scope.xlsx")

# =====================================================
# 3) MAJORITY VOTING SENTIMEN DAN LABEL FINAL
# =====================================================
voting_df["sentiment"]=voting_df[sentiment_annotation_cols].apply(lambda r: majority_vote(list(r)),axis=1)
rows_before_majority_filter=len(voting_df)
no_majority_mask=voting_df[["aspect","sentiment"]].isna().any(axis=1)
no_majority_df=voting_df.loc[no_majority_mask].copy(); majority_df=voting_df.loc[~no_majority_mask].copy()
if not no_majority_df.empty: save_df(no_majority_df,"06_excluded_no_majority.xlsx")
labeled_df=majority_df[majority_df["aspect"].isin(ASPECT_MODEL_LABELS)&majority_df["sentiment"].isin(SENTIMENT_LABELS)].copy()
labeled_df["joint_label"]=np.where(labeled_df["aspect"]==IRRELEVANT_ASPECT_LABEL,IRRELEVANT_ASPECT_LABEL,labeled_df["aspect"].astype(str)+"__"+labeled_df["sentiment"].astype(str))
labeled_df["split_label"]=labeled_df["joint_label"]
irrelevant_df=labeled_df[labeled_df["aspect"]==IRRELEVANT_ASPECT_LABEL].copy()
relevant_labeled_df=labeled_df[labeled_df["aspect"].isin(RESEARCH_ASPECT_LABELS)].copy()
save_df(annotation_df,"06_annotation_labels_before_filter.xlsx")
save_df(irrelevant_df,"06_irrelevant_aspect_retained_for_aspect_model.xlsx")
save_df(labeled_df,"06_final_majority_labels.xlsx")
labeling_quality_df=pd.DataFrame([{
 'rows_master_sample':len(master_df),'rows_valid_for_majority_voting':len(kappa_input_df),'rows_excluded_invalid_or_missing':len(kappa_excluded_df),'rows_used_kappa_aspect':len(aspect_kappa_input_df),'rows_used_kappa_sentiment_relevant_only':len(sentiment_kappa_input_df),'rows_excluded_kappa_sentiment_outside_scope':sentiment_rows_excluded_outside_scope,'rows_before_majority_filter':rows_before_majority_filter,'rows_excluded_no_majority':len(no_majority_df),'rows_after_majority':len(labeled_df),'rows_relevant_for_sentiment_model':len(relevant_labeled_df),'rows_irrelevant_retained_for_aspect_model':len(irrelevant_df),'fleiss_kappa_aspect':kappa_aspect,'fleiss_kappa_sentiment_relevant_only':kappa_sentiment,'aspect_policy':ASPECT_LABELING_POLICY,'annotation_provenance':ANNOTATION_PROVENANCE,'final_research_run':ANNOTATION_PROVENANCE in {"manual_human","ai_assisted_human_verified"},'annotation_note':'tidak_relevan dipertahankan pada model aspek; Kappa sentimen dan model sentimen hanya memakai empat aspek penelitian.'
}])
display(labeling_quality_df); save_df(labeling_quality_df,"06_labeling_quality_report.xlsx")
with open(OUT_DIR/"06_annotation_provenance.json","w",encoding="utf-8") as f:
 json.dump({'annotation_provenance':ANNOTATION_PROVENANCE,'final_research_run':bool(ANNOTATION_PROVENANCE in {"manual_human","ai_assisted_human_verified"}),'kappa_scope':{'aspect':'seluruh_baris_dengan_tiga_label_aspek_lengkap','sentiment':'hanya_majority_aspect_relevan'}},f,indent=2,ensure_ascii=False)

# =====================================================
# 3) DISTRIBUSI LABEL
# =====================================================
aspect_counts = (
    labeled_df["aspect"]
    .value_counts()
    .reindex(ASPECT_MODEL_LABELS, fill_value=0)
)
relevant_sentiment_counts = (
    relevant_labeled_df["sentiment"]
    .value_counts()
    .reindex(SENTIMENT_LABELS, fill_value=0)
)
joint_model_counts = (
    labeled_df["joint_label"]
    .value_counts()
    .sort_index()
)

display(aspect_counts.to_frame("count"))
display(relevant_sentiment_counts.to_frame("count"))
display(joint_model_counts.to_frame("count"))

plot_count_series(
    aspect_counts,
    "Distribusi Label Aspek Termasuk Kelas Tidak Relevan",
    "Kelas model aspek",
    "Jumlah",
    "06_distribusi_aspek_label_5kelas.png",
)
plot_count_series(
    relevant_sentiment_counts,
    "Distribusi Label Sentimen pada Ulasan Relevan",
    "Sentimen",
    "Jumlah",
    "06_distribusi_sentimen_relevan.png",
    rotation=0,
)

joint_table = (
    pd.crosstab(
        relevant_labeled_df["aspect"],
        relevant_labeled_df["sentiment"],
    )
    .reindex(
        index=RESEARCH_ASPECT_LABELS,
        columns=SENTIMENT_LABELS,
        fill_value=0,
    )
)
display(joint_table)
save_df(joint_table.reset_index(), "06_joint_label_table.xlsx")

joint_model_distribution_df = (
    joint_model_counts
    .rename_axis("joint_label")
    .reset_index(name="count")
)
save_df(
    joint_model_distribution_df,
    "06_joint_model_target_distribution.xlsx",
)
plot_heatmap_table(
    joint_table,
    "Heatmap Joint Label Aspek x Sentimen pada Data Relevan",
    "06_heatmap_joint_label_relevan.png",
)

if ANNOTATION_PROVENANCE == "synthetic_ai_pipeline_test":
    print(
        "\nPERINGATAN AKADEMIK: hasil ini hanya valid sebagai uji pipeline. "
        "Ganti provenance dan gunakan file yang benar-benar diverifikasi manusia "
        "sebelum menghasilkan hasil final skripsi."
    )


## 7. Group Split dan Random Oversampling

Perbaikan metodologis pada tahap ini:

1. Ulasan yang identik atau sangat mirip dikelompokkan menggunakan normalisasi teks dan kemiripan TF-IDF karakter.
2. Pembagian data menggunakan `StratifiedGroupKFold` lima fold:
   - tiga fold untuk training;
   - satu fold untuk validation;
   - satu fold untuk testing.
3. Satu kelompok ulasan mirip tidak boleh tersebar ke subset berbeda.
4. *Random Oversampling* hanya dilakukan setelah split:
   - model aspek diseimbangkan berdasarkan lima kelas aspek;
   - model sentimen hanya memakai empat aspek relevan dan diseimbangkan berdasarkan kombinasi aspek–sentimen;
   - tidak ada undersampling kelas mayoritas.


In [ ]:
# =========================================================
# 8. NEAR-DUPLICATE GROUP SPLIT DAN RANDOM OVERSAMPLING
# =========================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedGroupKFold


def create_group_text(text):
    """Normalisasi khusus untuk mendeteksi ulasan template/near-duplicate.

    Fungsi ini tidak mengganti review_clean yang masuk model.
    """
    text = str(text).lower()
    text = re.sub(
        r"\b(?:username|user\s*name|userid|user\s*id)\b\s*[:\-]?\s*\w+",
        " ",
        text,
    )
    text = re.sub(r"\b(?:aset\s*masa\s*depan|asetmasadepan)\b", " ", text)
    text = re.sub(r"\b\d+(?:[.,]\d+)?\b", " NUM ", text)
    text = re.sub(r"\b(?:rp|idr)\s*NUM\b", " MONEY ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size

    def find(self, item):
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, left, right):
        root_left = self.find(left)
        root_right = self.find(right)
        if root_left == root_right:
            return
        if self.rank[root_left] < self.rank[root_right]:
            root_left, root_right = root_right, root_left
        self.parent[root_right] = root_left
        if self.rank[root_left] == self.rank[root_right]:
            self.rank[root_left] += 1


def build_near_duplicate_groups(df):
    working = df.copy().reset_index(drop=True)
    working["group_text"] = working["review_clean"].apply(create_group_text)
    empty_mask = working["group_text"].str.len() == 0
    working.loc[empty_mask, "group_text"] = working.loc[
        empty_mask, "review_clean"
    ].astype(str)

    union_find = UnionFind(len(working))

    # Gabungkan teks yang sama setelah normalisasi template.
    first_index_by_text = {}
    for row_index, text in enumerate(working["group_text"]):
        if text in first_index_by_text:
            union_find.union(row_index, first_index_by_text[text])
        else:
            first_index_by_text[text] = row_index

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=NEAR_DUP_MAX_FEATURES,
        sublinear_tf=True,
    )
    matrix = vectorizer.fit_transform(working["group_text"])

    neighbor_count = min(NEAR_DUP_N_NEIGHBORS, len(working))
    neighbors = NearestNeighbors(
        n_neighbors=neighbor_count,
        metric="cosine",
        algorithm="brute",
        n_jobs=-1,
    )
    neighbors.fit(matrix)
    distances, indices = neighbors.kneighbors(matrix)

    similar_pair_rows = []
    for source_index in range(len(working)):
        for distance, target_index in zip(
            distances[source_index],
            indices[source_index],
        ):
            target_index = int(target_index)
            if target_index <= source_index:
                continue
            similarity = 1.0 - float(distance)
            if similarity >= NEAR_DUP_SIMILARITY_THRESHOLD:
                union_find.union(source_index, target_index)
                similar_pair_rows.append({
                    "source_row": source_index,
                    "target_row": target_index,
                    "source_id": working.loc[source_index, "id"],
                    "target_id": working.loc[target_index, "id"],
                    "similarity": similarity,
                    "source_text": working.loc[source_index, "review_clean"],
                    "target_text": working.loc[target_index, "review_clean"],
                })

    roots = [union_find.find(index) for index in range(len(working))]
    unique_roots = {
        root: group_number
        for group_number, root in enumerate(sorted(set(roots)), start=1)
    }
    working["near_dup_group"] = [
        f"G{unique_roots[root]:06d}" for root in roots
    ]

    pair_df = pd.DataFrame(similar_pair_rows)
    group_audit = (
        working.groupby("near_dup_group")
        .agg(
            group_size=("id", "size"),
            unique_split_labels=("split_label", "nunique"),
            representative_id=("id", "first"),
            representative_text=("review_clean", "first"),
        )
        .reset_index()
        .sort_values(["group_size", "near_dup_group"], ascending=[False, True])
    )

    return working, pair_df, group_audit


labeled_df, near_duplicate_pairs_df, near_duplicate_group_audit_df = (
    build_near_duplicate_groups(labeled_df)
)

save_df(
    labeled_df[
        [
            "id",
            "review_clean",
            "aspect",
            "sentiment",
            "joint_label",
            "split_label",
            "group_text",
            "near_dup_group",
        ]
    ],
    "07_labeled_data_with_near_duplicate_groups.xlsx",
)
save_df(
    near_duplicate_pairs_df,
    "07_near_duplicate_pairs_above_threshold.xlsx",
)
save_df(
    near_duplicate_group_audit_df,
    "07_near_duplicate_group_audit.xlsx",
)

print("Jumlah data berlabel:", len(labeled_df))
print("Jumlah near-duplicate group:", labeled_df["near_dup_group"].nunique())
print(
    "Jumlah pasangan dengan similarity >=",
    NEAR_DUP_SIMILARITY_THRESHOLD,
    ":",
    len(near_duplicate_pairs_df),
)
display(near_duplicate_group_audit_df.head(20))

# =========================================================
# STRATIFIED GROUP SPLIT 60:20:20
# =========================================================
class_group_counts = (
    labeled_df.groupby("split_label")["near_dup_group"]
    .nunique()
    .sort_values()
    .rename("unique_group_count")
    .reset_index()
)
display(class_group_counts)
save_df(class_group_counts, "07_class_group_count_before_split.xlsx")

minimum_group_count = int(class_group_counts["unique_group_count"].min())
if minimum_group_count < GROUP_SPLIT_N_FOLDS:
    raise ValueError(
        "StratifiedGroupKFold lima fold tidak aman karena terdapat kelas "
        f"dengan hanya {minimum_group_count} kelompok unik. "
        "Tambahkan data atau periksa threshold near-duplicate."
    )

splitter = StratifiedGroupKFold(
    n_splits=GROUP_SPLIT_N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

fold_assignment = np.full(len(labeled_df), -1, dtype=int)
for fold_index, (_, holdout_indices) in enumerate(
    splitter.split(
        X=np.zeros(len(labeled_df)),
        y=labeled_df["split_label"],
        groups=labeled_df["near_dup_group"],
    )
):
    fold_assignment[holdout_indices] = fold_index

if (fold_assignment < 0).any():
    raise RuntimeError("Ada data yang belum mendapat fold.")

labeled_df["group_fold"] = fold_assignment

# Fold 0 = test, fold 1 = validation, fold 2–4 = training.
test_df = labeled_df[labeled_df["group_fold"] == 0].copy()
val_df = labeled_df[labeled_df["group_fold"] == 1].copy()
train_df_raw = labeled_df[labeled_df["group_fold"].isin([2, 3, 4])].copy()

train_df_raw = train_df_raw.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = val_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Audit group leakage.
train_groups = set(train_df_raw["near_dup_group"])
val_groups = set(val_df["near_dup_group"])
test_groups = set(test_df["near_dup_group"])

group_overlap_audit_df = pd.DataFrame([
    {
        "pair": "train_vs_validation",
        "overlap_group_count": len(train_groups & val_groups),
    },
    {
        "pair": "train_vs_test",
        "overlap_group_count": len(train_groups & test_groups),
    },
    {
        "pair": "validation_vs_test",
        "overlap_group_count": len(val_groups & test_groups),
    },
])
display(group_overlap_audit_df)
save_df(group_overlap_audit_df, "07_group_split_overlap_audit.xlsx")
assert group_overlap_audit_df["overlap_group_count"].sum() == 0, (
    "Near-duplicate group tersebar ke subset berbeda."
)

# Audit ID overlap.
train_ids = set(train_df_raw["id"])
val_ids = set(val_df["id"])
test_ids = set(test_df["id"])
id_overlap_audit_df = pd.DataFrame([
    {"pair": "train_vs_validation", "overlap_id_count": len(train_ids & val_ids)},
    {"pair": "train_vs_test", "overlap_id_count": len(train_ids & test_ids)},
    {"pair": "validation_vs_test", "overlap_id_count": len(val_ids & test_ids)},
])
display(id_overlap_audit_df)
save_df(id_overlap_audit_df, "07_id_split_overlap_audit.xlsx")
assert id_overlap_audit_df["overlap_id_count"].sum() == 0

split_summary_df = pd.DataFrame([
    {
        "subset": "train_raw",
        "rows": len(train_df_raw),
        "groups": train_df_raw["near_dup_group"].nunique(),
        "ratio_actual": len(train_df_raw) / len(labeled_df),
    },
    {
        "subset": "validation",
        "rows": len(val_df),
        "groups": val_df["near_dup_group"].nunique(),
        "ratio_actual": len(val_df) / len(labeled_df),
    },
    {
        "subset": "test",
        "rows": len(test_df),
        "groups": test_df["near_dup_group"].nunique(),
        "ratio_actual": len(test_df) / len(labeled_df),
    },
])
display(split_summary_df)
save_df(split_summary_df, "07_split_summary_ratio.xlsx")

split_info = (
    labeled_df.groupby(["split_label", "group_fold"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={
        0: "test",
        1: "validation",
        2: "train_fold_2",
        3: "train_fold_3",
        4: "train_fold_4",
    })
    .reset_index()
)
for col in ["train_fold_2", "train_fold_3", "train_fold_4"]:
    if col not in split_info.columns:
        split_info[col] = 0
split_info["train"] = (
    split_info["train_fold_2"]
    + split_info["train_fold_3"]
    + split_info["train_fold_4"]
)
split_info["total"] = (
    split_info["train"]
    + split_info.get("validation", 0)
    + split_info.get("test", 0)
)
display(split_info)
save_df(split_info, "07_split_info_before_oversampling.xlsx")
save_df(train_df_raw, "07_train_raw.xlsx")
save_df(val_df, "07_validation.xlsx")
save_df(test_df, "07_test.xlsx")

print("Train raw:", train_df_raw.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# =========================================================
# RANDOM OVERSAMPLING — DATA LATIH SAJA
# =========================================================
def adaptive_target(counts, minimum, maximum):
    median_count = int(round(float(counts.median())))
    return int(np.clip(median_count, minimum, maximum))


def random_oversample_by_label(df, label_col, target_count, task_name):
    parts = []
    records = []

    for label_value, group in df.groupby(label_col, sort=True):
        group = group.copy()
        before = len(group)

        if before < target_count:
            additional_n = target_count - before
            random_state = SEED + zlib.crc32(
                f"{task_name}|{label_value}".encode("utf-8")
            )
            additional = group.sample(
                n=additional_n,
                replace=True,
                random_state=random_state,
            )
            sampled = pd.concat([group, additional], ignore_index=True)
            strategy = "random_oversample"
        else:
            additional_n = 0
            sampled = group
            strategy = "unchanged"

        parts.append(sampled)
        records.append({
            "task": task_name,
            "label_column": label_col,
            "label": label_value,
            "before": before,
            "target": target_count,
            "added_rows": additional_n,
            "after": len(sampled),
            "strategy": strategy,
        })

    output = (
        pd.concat(parts, ignore_index=True)
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )
    return output, pd.DataFrame(records)


# Dataset model aspek: lima kelas, termasuk tidak_relevan.
aspect_counts_before = train_df_raw["aspect"].value_counts()
ASPECT_ROS_TARGET = adaptive_target(
    aspect_counts_before,
    ASPECT_ROS_TARGET_MIN,
    ASPECT_ROS_TARGET_MAX,
)
train_aspect_df, aspect_balance_info = random_oversample_by_label(
    train_df_raw,
    label_col="aspect",
    target_count=ASPECT_ROS_TARGET,
    task_name="aspect_model_5_class",
)

# Dataset model sentimen: hanya ulasan relevan.
train_sentiment_raw_df = train_df_raw[
    train_df_raw["aspect"].isin(RESEARCH_ASPECT_LABELS)
].copy()

sentiment_joint_counts_before = train_sentiment_raw_df[
    "joint_label"
].value_counts()
SENTIMENT_JOINT_ROS_TARGET = adaptive_target(
    sentiment_joint_counts_before,
    SENTIMENT_JOINT_ROS_TARGET_MIN,
    SENTIMENT_JOINT_ROS_TARGET_MAX,
)
train_sentiment_df, sentiment_balance_info = random_oversample_by_label(
    train_sentiment_raw_df,
    label_col="joint_label",
    target_count=SENTIMENT_JOINT_ROS_TARGET,
    task_name="sentiment_model_relevant_joint",
)

balance_info = pd.concat(
    [aspect_balance_info, sentiment_balance_info],
    ignore_index=True,
)

display(pd.DataFrame([{
    "aspect_ros_target": ASPECT_ROS_TARGET,
    "sentiment_joint_ros_target": SENTIMENT_JOINT_ROS_TARGET,
    "train_raw_rows": len(train_df_raw),
    "train_aspect_rows_after_ros": len(train_aspect_df),
    "train_sentiment_relevant_raw_rows": len(train_sentiment_raw_df),
    "train_sentiment_rows_after_ros": len(train_sentiment_df),
}]))
display(balance_info)

save_df(balance_info, "07_oversampling_info_train.xlsx")
save_df(train_aspect_df, "07_train_aspect_oversampled.xlsx")
save_df(train_sentiment_df, "07_train_sentiment_oversampled.xlsx")

# Alias kompatibilitas; isinya dataset model aspek.
save_df(train_aspect_df, "07_train_oversampled.xlsx")

# Assertion tidak ada undersampling dan data asli tidak hilang.
assert (aspect_balance_info["after"] >= aspect_balance_info["before"]).all()
assert (sentiment_balance_info["after"] >= sentiment_balance_info["before"]).all()
assert set(train_df_raw["id"]).issubset(set(train_aspect_df["id"]))
assert set(train_sentiment_raw_df["id"]).issubset(
    set(train_sentiment_df["id"])
)
assert set(train_aspect_df["aspect"].unique()).issubset(
    set(ASPECT_MODEL_LABELS)
)
assert set(train_sentiment_df["aspect"].unique()).issubset(
    set(RESEARCH_ASPECT_LABELS)
)

print("Target ROS model aspek:", ASPECT_ROS_TARGET)
print("Target ROS joint model sentimen:", SENTIMENT_JOINT_ROS_TARGET)
print("Train model aspek:", train_aspect_df.shape)
print("Train model sentimen:", train_sentiment_df.shape)


## 7A. Audit Split, Near-Duplicate, dan Random Oversampling

Bagian ini menampilkan:

- jumlah kelompok ulasan mirip;
- audit tidak adanya kelompok yang bocor antar-subset;
- distribusi lima kelas model aspek sebelum dan sesudah ROS;
- distribusi joint label model sentimen sebelum dan sesudah ROS;
- target oversampling adaptif yang digunakan.


In [ ]:
# =========================================================
# 8A. VISUALISASI DAN AUDIT METODOLOGI
# =========================================================

def make_distribution_table(before_series, after_series, label_name):
    before = before_series.value_counts().sort_index().rename("before")
    after = after_series.value_counts().sort_index().rename("after")
    output = pd.concat([before, after], axis=1).fillna(0).astype(int)
    output["difference"] = output["after"] - output["before"]
    return output.reset_index().rename(columns={"index": label_name})


def plot_before_after_distribution(table_df, label_col, title, filename):
    plot_df = table_df.set_index(label_col)[["before", "after"]]
    ax = plot_df.plot(kind="bar", figsize=(11, 5))
    plt.title(title)
    plt.xlabel(label_col)
    plt.ylabel("Jumlah data")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    path = FIG_DIR / filename
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved figure:", path)


aspect_balance_table = make_distribution_table(
    train_df_raw["aspect"],
    train_aspect_df["aspect"],
    "aspect",
)
sentiment_joint_balance_table = make_distribution_table(
    train_sentiment_raw_df["joint_label"],
    train_sentiment_df["joint_label"],
    "joint_label",
)

# Alias untuk bagian BAB IV lama.
joint_balance_table = sentiment_joint_balance_table.copy()

display(aspect_balance_table)
display(sentiment_joint_balance_table)

save_df(
    aspect_balance_table,
    "07A_aspect_distribution_before_after_oversampling.xlsx",
)
save_df(
    sentiment_joint_balance_table,
    "07A_joint_label_distribution_before_after_oversampling.xlsx",
)

plot_before_after_distribution(
    aspect_balance_table,
    "aspect",
    "Model Aspek: Sebelum vs Sesudah Random Oversampling",
    "07A_aspect_distribution_before_after_oversampling.png",
)
plot_before_after_distribution(
    sentiment_joint_balance_table,
    "joint_label",
    "Model Sentimen: Joint Label Sebelum vs Sesudah Random Oversampling",
    "07A_joint_label_distribution_before_after_oversampling.png",
)

imbalance_summary = pd.DataFrame([
    {
        "task": "aspect_model_5_class",
        "target": ASPECT_ROS_TARGET,
        "before_min": int(train_df_raw["aspect"].value_counts().min()),
        "before_max": int(train_df_raw["aspect"].value_counts().max()),
        "after_min": int(train_aspect_df["aspect"].value_counts().min()),
        "after_max": int(train_aspect_df["aspect"].value_counts().max()),
    },
    {
        "task": "sentiment_model_relevant_joint",
        "target": SENTIMENT_JOINT_ROS_TARGET,
        "before_min": int(
            train_sentiment_raw_df["joint_label"].value_counts().min()
        ),
        "before_max": int(
            train_sentiment_raw_df["joint_label"].value_counts().max()
        ),
        "after_min": int(
            train_sentiment_df["joint_label"].value_counts().min()
        ),
        "after_max": int(
            train_sentiment_df["joint_label"].value_counts().max()
        ),
    },
])
display(imbalance_summary)
save_df(imbalance_summary, "07A_imbalance_summary.xlsx")

near_duplicate_summary_df = pd.DataFrame([{
    "labeled_rows": len(labeled_df),
    "near_duplicate_groups": labeled_df["near_dup_group"].nunique(),
    "groups_with_multiple_rows": int(
        (near_duplicate_group_audit_df["group_size"] > 1).sum()
    ),
    "largest_group_size": int(
        near_duplicate_group_audit_df["group_size"].max()
    ),
    "similarity_threshold": NEAR_DUP_SIMILARITY_THRESHOLD,
    "pair_count_above_threshold": len(near_duplicate_pairs_df),
    "group_overlap_total": int(
        group_overlap_audit_df["overlap_group_count"].sum()
    ),
}])
display(near_duplicate_summary_df)
save_df(
    near_duplicate_summary_df,
    "07A_near_duplicate_methodology_summary.xlsx",
)

assert (aspect_balance_table["difference"] >= 0).all()
assert (sentiment_joint_balance_table["difference"] >= 0).all()
assert near_duplicate_summary_df["group_overlap_total"].iloc[0] == 0


In [ ]:
# =========================================================
# CHECKPOINT TAHAP 2 — V5.4 METODOLOGI
# =========================================================

required_stage2_files = [
    "06_final_majority_labels.xlsx",
    "06_fleiss_kappa.xlsx",
    "06_labeling_quality_report.xlsx",
    "06_annotation_provenance.json",
    "06_joint_label_table.xlsx",
    "07_labeled_data_with_near_duplicate_groups.xlsx",
    "07_near_duplicate_group_audit.xlsx",
    "07_group_split_overlap_audit.xlsx",
    "07_train_raw.xlsx",
    "07_train_aspect_oversampled.xlsx",
    "07_train_sentiment_oversampled.xlsx",
    "07_validation.xlsx",
    "07_test.xlsx",
    "07_split_info_before_oversampling.xlsx",
    "07_oversampling_info_train.xlsx",
    "07A_aspect_distribution_before_after_oversampling.xlsx",
    "07A_joint_label_distribution_before_after_oversampling.xlsx",
]

missing_stage2 = [
    filename
    for filename in required_stage2_files
    if not (STAGE2_DIR / filename).exists()
]
if missing_stage2:
    raise RuntimeError(
        "Checkpoint Tahap 2 belum lengkap: " + ", ".join(missing_stage2)
    )

stage2_manifest = pd.DataFrame([
    {"file": filename, "status": "tersedia"}
    for filename in required_stage2_files
])
stage2_manifest.to_excel(
    STAGE2_DIR / "STAGE2_MANIFEST.xlsx",
    index=False,
)

import zipfile
stage2_zip=Path(shutil.make_archive(str(PROJECT_DIR/"stage2_checkpoint_v5_4"),"zip",root_dir=STAGE2_DIR))
if not zipfile.is_zipfile(stage2_zip): raise RuntimeError(f"Checkpoint bukan ZIP valid: {stage2_zip}")
with zipfile.ZipFile(stage2_zip,"r") as archive: bad_member=archive.testzip()
if bad_member is not None: raise RuntimeError(f"Checkpoint Fase 2 rusak: {bad_member}")
print("Checkpoint Tahap 2 valid:",stage2_zip)
print("\nTAHAP 2 SELESAI.")
print(
    "Notebook Bagian 1 tidak perlu dijalankan ulang. "
    "Lanjutkan ke Notebook Bagian 3 untuk training dan evaluasi."
)
